# Benchmarki

# Symulowana Bifurkacja


In [3]:
import os
import time

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np

from funkcje_pomocnicze import read_instance, pegasus
from benchmarks import discrete_simulated_bifurcation_gpu, discrete_simulated_bifurcation, discrete_simulated_bifurcation_gpu_naive
from IPython.utils.io import capture_output
from tqdm import tqdm

P2 = pegasus(os.path.join("instancje", "Pegasus", "P2_CBFM-P.txt"), -39.0)
P4 = pegasus(os.path.join("instancje", "Pegasus", "P4_CBFM-P.txt"), -469.0)
P8 = pegasus(os.path.join("instancje", "Pegasus", "P8_CBFM-P.txt"), -2752.0)
#P12 = pegasus(os.path.join("instancje", "Pegasus", "P12_CBFM-P.txt"), -6831.0)
P16 = pegasus(os.path.join("instancje", "Pegasus", "P16_CBFM-P.txt"), -12772.0)  


functions = [discrete_simulated_bifurcation_gpu]

global_results = []
for function in functions:
    gaps = []
    times = []
    for instance in tqdm([P2, P4, P8]):
        J, h = read_instance(instance.path, convention="minus_half")
        J2 = cp.asarray(J, dtype=cp.float32)
        h2 = cp.asarray(h, dtype=cp.float32)
        # kompilacja 
        with capture_output() as captured:
            states, energies = function(J2, h2, 100, 0.25, 2**5)

        result = 0
        elapsed = 0
        count = 0
        start = time.time()
        while result != instance.best_energy or elapsed >= 600:
            with capture_output() as captured:
                states, energies = function(J2, h2, 1000 * (1 + count), 0.5, 2**(10 + count))
            result = min(energies)
            checkpoint = time.time()
            elapsed = checkpoint - start
            count += 1
        print(elapsed)

        gaps.append((instance.best_energy - min(energies).item())/instance.best_energy)      
        times.append(elapsed) 


instancje = ["P2", "P4", "P8"]

x = np.arange(len(instancje))
width = 0.25  

fig, ax = plt.subplots(figsize=(12, 6))

#rects2 = ax.bar(x, global_results[1], width, label='GPU naiwne')
rects3 = ax.bar(x + width, global_results[0], width, label='GPU kernel')

ax.set_xlabel("Rozmiary instancji")
ax.set_ylabel("Czas do znalezienia rozwiązania")
ax.set_title('Czas do znalezienia rozwiązania')
ax.set_xticks(x)
ax.set_xticklabels(instancje, rotation=45)
ax.legend()

plt.tight_layout()
plt.show()


 33%|███▎      | 1/3 [00:00<00:00,  3.49it/s]

0.1875152587890625


 67%|██████▋   | 2/3 [00:00<00:00,  3.58it/s]

0.2072286605834961


 67%|██████▋   | 2/3 [01:21<00:40, 40.84s/it]


KeyboardInterrupt: 

In [ ]:
import torch
import simulated_bifurcation as sb
import numpy as np
from funkcje_pomocnicze import read_instance, pegasus_benchmark, test_pegasus, ising_to_qubo, small_pegasus
from benchmarks import discrete_simulated_bifurcation

J, h = read_instance(small_pegasus.path, convention="minus_half")
J1, h1 = read_instance(small_pegasus.path, convention="dwave")

J1 = torch.tensor(J1, dtype=torch.float32)
h1 = torch.tensor(h1, dtype=torch.float32)




states, energies = discrete_simulated_bifurcation(J, h, 200, 0.25, 2**9)
print(min(energies))
sb.minimize(J1, h1, domain="spin", dtype=torch.float32)

Symulowana Bifurkacja: 100%|██████████| 200/200 [00:01<00:00, 120.10it/s]


-39.0


🏁 Bifurcated agents: 100%|██████████| 128/128 [00:00<00:00, 455.58 agents/s]


(tensor([-1., -1., -1., -1.,  1.,  1.,  1.,  1.,  1., -1., -1., -1.,  1., -1.,
          1.,  1., -1.,  1.,  1.,  1.,  1.,  1., -1.,  1.]),
 tensor(-33.))

In [1]:
import torch
import numpy as np
import simulated_bifurcation as sb

matrix = torch.tensor(
    [
        [0, 1, -1],
        [1, 0, 2],
        [-1, 2, 0],
    ],
    dtype=torch.float32,
    device=torch.device("cuda")
)

print(matrix.device)

sb.minimize(matrix, dtype=torch.float32, device=torch.device("cuda"))

cuda:0


🔁 Iterations       :   0%|          | 0/10000 [00:00<?, ? steps/s]

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

## Implementacja GPU

In [ ]:
import torch
import simulated_bifurcation as sb
import cupy as cp
from funkcje_pomocnicze import read_instance,  pegasus_benchmark
from benchmarks import discrete_simulated_bifurcation_gpu
from IPython.utils.io import capture_output

for instance in pegasus_benchmark:
    J, h = read_instance(instance.path, convention="minus_half")
    J2 = cp.asarray(J, dtype=cp.float32)
    h2 = cp.asarray(h, dtype=cp.float32)
    
    with capture_output() as captured:
        states, energies = discrete_simulated_bifurcation_gpu(J2, h2, 1000, 0.25, 2**11)

    print(min(energies))
    print(instance.best_energy)

# Brute Force

In [3]:
from funkcje_pomocnicze import read_instance, small_pegasus
from dimod import BinaryQuadraticModel
from dimod.serialization import coo

J, h = read_instance(small_pegasus.path, convention="dwave")

bqm = BinaryQuadraticModel(h, J, vartype="SPIN")

with open("small_pegasus_bqm.txt", "w") as f:
    coo.dump(bqm, f)

In [2]:
# Tworzenie instancji
import os
import numpy as np
from dimod import BinaryQuadraticModel
from dimod.serialization import coo

for n in range(2, 33):
    Q = np.random.uniform(-1, 1, size=(n, n))
    Q = np.triu(Q)
    bqm = BinaryQuadraticModel(Q, vartype="BINARY")
    with open(os.path.join("instancje", "bruteforce", f"przykład_{n}"), "w") as f:
        coo.dump(bqm, f, vartype_header=True)

In [ ]:
##############################
# Uruchom tą komurkę z kernelem "bruteforce"
##############################
#TODO: Inny sposób puszczania brutefdorce, za duży overhead w ten sposób
import os
import time
import numpy as np
from tqdm import tqdm

# kompilacja
command = "omnisolver bruteforce-gpu small_pegasus_bqm.txt --output test.txt --vartype SPIN --num_states 10"
os.system(command)
print("zkompilowane")
command = f"omnisolver bruteforce-gpu instancje/bruteforce/przykład_10 --output wyniki/przykład_3 --vartype BINARY --num_states 5"
os.system(command)

# times_mean = []
# for n in tqdm(range(2, 32)):
#     command = f"omnisolver bruteforce-gpu instancje/bruteforce/przykład_{n} --output wyniki/przykład_{n} --vartype BINARY --num_states 10"
#     times = []
#     for _ in range(5):
#         start = time.time()
#         os.system(command)
#         end = time.time()
#         times.append(end - start)
#     times_mean.append(np.mean(times))

# print(times_mean)
# with open("bruteforce_wyniki.txt", "w") as f:
#     f.write(f"{times_mean}")

zkompilowane
